In [1]:
import numpy as np
import pandas as pd
from gensim import matutils, corpora
import pickle

In [2]:
with open('filtered_LDA_models.pkl', 'rb') as f:
    filtered_LDA_models = pickle.load(f)

with open('filtered_corpus.pkl', 'rb') as f:
    filtered_corpus = pickle.load(f)

with open('../../Corpus/processed_corpus.pkl', 'rb') as f:
    processed_corpus = pickle.load(f)

filtered_dictionary = corpora.Dictionary.load('filtered_dictionary.gensim')

In [3]:
def relevance_keywords(model, corpus, dictionary=None, topn=10, lam=0.6, return_scores=False):
    # Always map with the model’s own dictionary
    model_dict = model.id2word
    # Use the model’s V for p(w)
    phi = model.get_topics()  # (K, V)
    K, V = phi.shape

    # p(w) over the same V space
    csc = matutils.corpus2csc(corpus, num_terms=V)
    term_freq = np.asarray(csc.sum(axis=1)).ravel().astype(float)
    term_freq[term_freq == 0.0] = 1e-12
    p_w = term_freq / term_freq.sum()

    log_phi = np.log(np.clip(phi, 1e-12, None))
    log_pw  = np.log(p_w)
    relevance = log_phi - (1.0 - lam) * log_pw[np.newaxis, :]

    # Safe accessor (works if id2token is dict or list-like)
    i2t = getattr(model_dict, "id2token", None)
    if i2t is None:
        inv = {v: k for k, v in model_dict.token2id.items()}
        def get_tok(i): return inv.get(i)
    elif isinstance(i2t, dict):
        def get_tok(i): return i2t.get(int(i))
    else:  # list-like
        def get_tok(i):
            i = int(i)
            return i2t[i] if 0 <= i < len(i2t) else None

    out = {}
    for k in range(K):
        order = np.argsort(-relevance[k])
        picked, scores = [], []
        for i in order:
            tok = get_tok(i)
            if tok is None:
                continue
            picked.append(tok)
            scores.append(float(relevance[k, int(i)]))
            if len(picked) == topn:
                break
        out[k] = list(zip(picked, scores)) if return_scores else picked
    return out

In [4]:
filtered_ideal_topic_num = 8
filtered_model = filtered_LDA_models[filtered_ideal_topic_num]
num_keywords = 10
lam = 0.6

rel_kws = relevance_keywords(
    filtered_model,
    filtered_corpus,
    filtered_dictionary,
    topn=num_keywords,
    lam=lam,
    return_scores=False  # set True if I want to inspect scores
)

In [5]:
print(f"Top {num_keywords} keywords per topic (lambda={lam}):")
for topic_id, keywords in rel_kws.items():
    print(f"Topic {topic_id + 1}: {', '.join(keywords)}")

Top 10 keywords per topic (lambda=0.6):
Topic 1: mensa, compono, aspectus, obsto, abscondo, cibus, abnuo, orno, hospitium, intactus
Topic 2: classis, nuntius, remeo, niueus, pudicus, memini, lacero, furtiuus, grates, nescius
Topic 3: subiectus, iulius, puellaris, propago, servo, adimo, sollers, ambitio, ostento, tremulus
Topic 4: bos, opimus, uinclum, cultus, classis, curuus, celo, bustum, torpeo, misereor
Topic 5: exclamo, afflicto, ornus, gestio, quaesitor, ile, linea, adortus, incruentus, impietas
Topic 6: princeps, subicio, constituo, suboles, consensus, extruo, uolucer, lacero, porto, incestus
Topic 7: uoltus, uindico, uolucer, axis, inuado, colus, intendo, glacialis, ursa, rigidus
Topic 8: conuexus, facultas, instigo, labes, saluo, abscisus, intonsus, inspiro, strepitus, procrustes


In [6]:
# Must run BEFORE building topic_to_acts
assert len(filtered_corpus) == len(processed_corpus), \
    f"Mismatch: corpus={len(filtered_corpus)} vs processed_corpus={len(processed_corpus)}"

K = filtered_model.num_topics
print(f"Num_topics K={K}, docs={len(filtered_corpus)}")

# Quick peek at one document distribution with dense probs:
tmp = filtered_model.get_document_topics(filtered_corpus[0], minimum_probability=0.0)
print(f"First doc dense dist len={len(tmp)} (expect {K})")


Num_topics K=8, docs=11
First doc dense dist len=8 (expect 8)


In [7]:
# --- Top 5 acts per topic -------------------------
K = filtered_model.num_topics #grab the number of topics K from my trained LDA model

# Recompute full distributions with minimum_probability=0.0
# so every topic gets a score for every act.
topic_to_acts = {k: [] for k in range(K)}      #Build a dictionary that maps each topic to an empty list
for i, bow in enumerate(filtered_corpus):  #Iterate through every act’s text and get its list of word-frequency pairs (BoW)
    full_dist = filtered_model.get_document_topics(bow, minimum_probability=0.0) #get full topic distribution for this act, minimum_prob=0.0 forces all topics to be included
                                                                                    #get_document_topics returns a list of (topic_id, weight) pairs
    title = processed_corpus[i]["title"]
    for k, w in full_dist:                 #loop through every (topic_id, weight) pair for this act
        topic_to_acts[k].append((title, float(w)))  #Append a tuple (act_id, weight) to the list for topic k

# Keep only the top 5 acts per topic, sorted by descending weight
top5_by_topic = {
    k: sorted(v, key=lambda t: -t[1])[:5]  #k is topic_id, v is list of (act_id, weight) tuples #for k, v in topic_to_acts.items()
                                            #each tuple t has t[0]=act_id, t[1]=weight; -t[1] sorts by negative weight (descending--largest first)
                                            #key=lambda t: -t[1] is the sorting rule, need to use sort() to actually do something with the rule
    for k, v in topic_to_acts.items()
}


In [8]:
for k in range(K):
    print(f"[TOP5] topic {k}: {top5_by_topic[k]}")

[TOP5] topic 0: [('Thyestes', 0.9985335469245911), ('Phoenissae', 0.997355043888092), ('Phaedra', 0.16421271860599518), ('Hercules Furens', 0.1361670196056366), ('Ecerinis', 0.00033107385388575494)]
[TOP5] topic 1: [('Agamemnon', 0.9480828046798706), ('Phaedra', 0.17331746220588684), ('Phoenissae', 0.00037789344787597656), ('Ecerinis', 0.00033103220630437136), ('Octavia', 0.00029495966737158597)]
[TOP5] topic 2: [('Phoenissae', 0.000377694028429687), ('Ecerinis', 0.0003307524311821908), ('Octavia', 0.0002948324545286596), ('Troades', 0.00023544326541014016), ('Medea', 0.0002323890512343496)]
[TOP5] topic 3: [('Oedipus', 0.9986754655838013), ('Troades', 0.998350977897644), ('Hercules Furens', 0.37186843156814575), ('Phaedra', 0.3280099928379059), ('Phoenissae', 0.00037800398422405124)]
[TOP5] topic 4: [('Phoenissae', 0.000377694028429687), ('Ecerinis', 0.0003307524311821908), ('Octavia', 0.0002948324545286596), ('Troades', 0.00023544326541014016), ('Medea', 0.00023238908033818007)]
[TOP

In [9]:
# Build the spreadsheet rows:
# Topic Number should match gensim indexing (+1 to go 1..K).
rows = []
for topic_id in range(filtered_ideal_topic_num):
    # Defensive: skip if model has fewer topics than expected
    if topic_id not in rel_kws:
        print(f"[WARN] rel_kws missing topic {topic_id}; skipping")
        continue

    # Tragedies ranked by this topic (up to 5)
    top_tragedies = top5_by_topic.get(topic_id, [])

    for j, token in enumerate(rel_kws[topic_id]):
        # For the first five rows within the topic, attach "title (weight)"
        title = ""
        if j < len(top_tragedies):
            title, w = top_tragedies[j]
            tragedy = f"{title} ({w:.3f})"

        rows.append({
            "Topic Number": topic_id + 1,       # 1-based label for readability
            "Tragedy": tragedy,                     
            "Latin Keywords": token,            # one per row
            "English Keywords": "",             # left empty
            "Themes": "",                       # left empty
            "Topic Name": ""                    # left empty
        })

df = pd.DataFrame(rows, columns=[
    "Topic Number", "Tragedy", "Latin Keywords", "English Keywords", "Themes", "Topic Name"
])



In [10]:
# Sanity checks:

print("Row keys sample:", rows[0].keys())

filled = [r for r in rows if r["Tragedy"]]
print(f"Rows with Tragedy filled: {len(filled)} of {len(rows)}")
print("Sample 5:", filled[:5])

nonempty_count = (df["Tragedy"] != "").sum()
print(f"DataFrame non-empty '{'Tragedy'}': {nonempty_count}")

Row keys sample: dict_keys(['Topic Number', 'Tragedy', 'Latin Keywords', 'English Keywords', 'Themes', 'Topic Name'])
Rows with Tragedy filled: 80 of 80
Sample 5: [{'Topic Number': 1, 'Tragedy': 'Thyestes (0.999)', 'Latin Keywords': 'mensa', 'English Keywords': '', 'Themes': '', 'Topic Name': ''}, {'Topic Number': 1, 'Tragedy': 'Phoenissae (0.997)', 'Latin Keywords': 'compono', 'English Keywords': '', 'Themes': '', 'Topic Name': ''}, {'Topic Number': 1, 'Tragedy': 'Phaedra (0.164)', 'Latin Keywords': 'aspectus', 'English Keywords': '', 'Themes': '', 'Topic Name': ''}, {'Topic Number': 1, 'Tragedy': 'Hercules Furens (0.136)', 'Latin Keywords': 'obsto', 'English Keywords': '', 'Themes': '', 'Topic Name': ''}, {'Topic Number': 1, 'Tragedy': 'Ecerinis (0.000)', 'Latin Keywords': 'abscondo', 'English Keywords': '', 'Themes': '', 'Topic Name': ''}]
DataFrame non-empty 'Tragedy': 80


In [11]:
csv_path = "../../Corpus/corpus_csv/8_topics_keywords_lambda06.csv"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} with {len(df)} rows.")

Wrote ../../Corpus/corpus_csv/8_topics_keywords_lambda06.csv with 80 rows.


In [12]:
filtered_document_topics = []
for i, bow in enumerate(filtered_corpus):
    topic_dist = filtered_model.get_document_topics(bow)  # topic_dist is a list of tuples representing the topic distribution for the i-th document # each tuple is (topic_id, topic_weight)
    filtered_document_topics.append((processed_corpus[i]["title"], topic_dist))

for title, topics in filtered_document_topics:
    # Get topic with highest probability
    top_topic = max(topics, key=lambda x: x[1])
    print(f"{title:<20} → Filtered Topic #{top_topic[0] + 1} (weight: {top_topic[1]:.3f})")
   
#get topic distribution for each play
for title, topics in filtered_document_topics:
    print(f"\n{title}")
    top_three = sorted(topics, key=lambda x: -x[1])[:3]
    for topic_id, weight in top_three:
        print(f"  Filtered Topic #{topic_id + 1}: {weight:.3f}")

Phoenissae           → Filtered Topic #1 (weight: 0.997)
Troades              → Filtered Topic #4 (weight: 0.998)
Phaedra              → Filtered Topic #7 (weight: 0.334)
Agamemnon            → Filtered Topic #2 (weight: 0.948)
Hercules Furens      → Filtered Topic #7 (weight: 0.491)
Medea                → Filtered Topic #7 (weight: 0.998)
Octavia              → Filtered Topic #6 (weight: 0.998)
Oedipus              → Filtered Topic #4 (weight: 0.999)
Thyestes             → Filtered Topic #1 (weight: 0.999)
Hercules Oetaeus     → Filtered Topic #7 (weight: 0.999)
Ecerinis             → Filtered Topic #7 (weight: 0.998)

Phoenissae
  Filtered Topic #1: 0.997

Troades
  Filtered Topic #4: 0.998

Phaedra
  Filtered Topic #7: 0.334
  Filtered Topic #4: 0.328
  Filtered Topic #2: 0.173

Agamemnon
  Filtered Topic #2: 0.948
  Filtered Topic #7: 0.051

Hercules Furens
  Filtered Topic #7: 0.491
  Filtered Topic #4: 0.372
  Filtered Topic #1: 0.136

Medea
  Filtered Topic #7: 0.998

Octavia
  